In [1]:
import sys, json, html, time
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_dataset


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from retrieval.dense import encode_texts, encode_queries
from eval.evaluation import score_retrieval

In [2]:
CORPUS_PATH = ROOT / "data" / "corpus" / "pooled_corpus.parquet"
SPLITS_DIR = ROOT / "splits"
GOLD_DIR = ROOT / "results" / "gold"
EMB_DIR = ROOT / "data" / "embeddings" / "contriever"
RUNS_DIR = ROOT / "data" / "runs"
EMB_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
corpus = pd.read_parquet(CORPUS_PATH, columns=["pid", "title", "text"])
corpus_texts = [html.unescape(title) + " " + text for title, text in zip(corpus.title, corpus.text)]
pids = corpus.pid.tolist()
pid_to_title = dict(zip(corpus.pid, corpus.title))

In [4]:
PASSAGES_PATH = EMB_DIR / "passages.npy"
if PASSAGES_PATH.exists():
    passages = np.load(PASSAGES_PATH)
    with open(EMB_DIR / "pids.json") as f:
        assert json.load(f) == pids, "saved embeddings do not match the corpus order"
else:
    start = time.time() 
    passages = encode_texts(corpus_texts, show_progress=True)
    print(f"Encoded {len(passages):,} passages in {(time.time() - start) / 60:.1f} min")
    np.save(PASSAGES_PATH, passages)
    with open(EMB_DIR / "pids.json", "w") as f:
        json.dump(pids, f)
    with open(EMB_DIR / "config.json", "w") as f:
        json.dump({"model": "facebook/contriever-msmarco", "pooling": "mean", "max_length": 256,
                    "similarity": "dot product", "text": "html.unescape(title) + ' ' + text",
                    "n_docs": len(pids), "dim": int(passages.shape[1])}, f, indent=2)
assert passages.shape == (len(corpus), 768) and not np.isnan(passages).any()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  0%|          | 0/3980 [00:00<?, ?it/s]

Encoded 509,315 passages in 11.7 min


In [5]:
SLICES = ["pilot", "tune"]
RETRIEVE_K = 60
KEEP_K = 50
CHUNK = 100   # questions per matrix product (100 x 509k scores at a time)

hotpot_ds = load_dataset("hotpotqa/hotpot_qa", "distractor")
train_q = hotpot_ds["train"].select_columns(["id", "question", "type"]).to_pandas().set_index("id")
qid_to_type = train_q["type"].to_dict()

def top_rows(query_vecs, passages, k=RETRIEVE_K, chunk=CHUNK):
    """Row indices of the k highest dot-product passages per query, best first."""
    out = []
    for start in range(0, len(query_vecs), chunk):
        scores = query_vecs[start:start + chunk] @ passages.T
        top = np.argpartition(-scores, k, axis=1)[:, :k]                 # unordered top k
        order = np.argsort(-np.take_along_axis(scores, top, axis=1), axis=1)
        out.append(np.take_along_axis(top, order, axis=1))
    return np.vstack(out)

def collapse_by_title(rows, pids, pid_to_title, keep=KEEP_K):
    """Ranked rows -> ranked pids, keeping only the best pid per title."""
    ranked, seen = [], set()
    for row in rows:
        pid = pids[row]
        title = pid_to_title[pid]
        if title not in seen:
            seen.add(title)
            ranked.append(pid)
            if len(ranked) == keep:
                break
    return ranked

for slice_name in SLICES:
    with open(SPLITS_DIR / f"{slice_name}.json") as f:
        ids = json.load(f)
    questions = train_q.loc[ids, "question"].tolist()

    query_vecs = encode_queries(questions)
    rows = top_rows(query_vecs, passages)
    run = {qid: collapse_by_title(r, pids, pid_to_title) for qid, r in zip(ids, rows)}

    assert list(run) == ids and all(len(r) == len(set(r)) <= KEEP_K for r in run.values())
    with open(RUNS_DIR / f"contriever_{slice_name}.json", "w") as f:
        json.dump(run, f)
    print(f"{slice_name}: {len(run)} questions saved")

pilot: 200 questions saved
tune: 1000 questions saved


In [6]:
for slice_name in SLICES:
    with open(GOLD_DIR / f"{slice_name}_gold.json") as f:
        gold = json.load(f)
    with open(RUNS_DIR / f"contriever_{slice_name}.json") as f:
        run = json.load(f)
    _, summary = score_retrieval(run, gold, qid_to_type, pid_to_title)
    print(f"\n=== Contriever, {slice_name} ===")
    print(pd.DataFrame(summary).T.round(2).to_string())


=== Contriever, pilot ===
                n  both@2  any@2  mean_gold@2  both@5   any@5  mean_gold@5  both@10  any@10  mean_gold@10  both@20  any@20  mean_gold@20  both@50  any@50  mean_gold@50
overall     200.0   24.00   88.0         1.12   40.50   93.00         1.34    44.50    94.0          1.38    53.50   95.50          1.49    59.50   97.00          1.56
bridge      160.0   13.12   85.0         0.98   29.38   91.25         1.21    33.12    92.5          1.26    41.88   94.38          1.36    49.38   96.25          1.46
comparison   40.0   67.50  100.0         1.68   85.00  100.00         1.85    90.00   100.0          1.90   100.00  100.00          2.00   100.00  100.00          2.00

=== Contriever, tune ===
                 n  both@2  any@2  mean_gold@2  both@5  any@5  mean_gold@5  both@10  any@10  mean_gold@10  both@20  any@20  mean_gold@20  both@50  any@50  mean_gold@50
overall     1000.0   22.10  86.30         1.08   37.80  91.20         1.29    47.00   93.80          1.41  